In [49]:
# 首先在代码开头添加修改版的函数定义
import os
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

In [ ]:

# 修改版的mixMaxScale_dff函数
def mixMaxScale_dff(df, feature_range=(0, 1)):
    '''
    改进版本，包含数据验证
    '''
    
    # 添加数据验证
    if df is None or df.empty:
        print("警告: 输入数据为空，返回原始数据")
        return df
    
    # 检查数据形状
    print(f"数据形状: {df.shape}")
    
    # 尝试转换为数值类型
    try:
        df_numeric = df.apply(pd.to_numeric, errors='coerce')
        
        # 检查是否有全为NaN的列
        nan_columns = df_numeric.columns[df_numeric.isna().all()].tolist()
        if nan_columns:
            print(f"警告: 以下列全为NaN值: {nan_columns}")
            
        # 删除全为NaN的列
        if nan_columns:
            df_numeric = df_numeric.drop(columns=nan_columns)
        
        # 检查剩余数据是否为空
        if df_numeric.empty:
            print("错误: 转换后数据为空")
            return df
        
        scaler = MinMaxScaler(feature_range=feature_range)
        scaler.fit(df_numeric.copy())
        
        dfN = pd.DataFrame(scaler.transform(df_numeric))
        dfN.columns = df_numeric.columns
        
        return dfN
        
    except Exception as e:
        print(f"缩放过程中发生错误: {e}")
        if hasattr(df, 'head'):
            print(f"数据前几行:\n{df.head()}")
        return df

# 修改版的get_BonsetA函数，添加调试信息
def get_BonsetA_debug(df,
                    which_status_to_check,
                    fps,
                    secForward=10,
                    secBackward=10,
                    beforeB=False,baseSecond=2,intervalLength=20,
                    if_CombineAdjacentChunks=True,gapMax=10,ifNorm=True):
    
    from Functions_plot_All_20250807 import check_frame_interval, find_compositeIntervals, normBA
    
    '''
    修改版，添加调试信息
    '''
    
    df_status = check_frame_interval(df, which_status_to_check, ifKeep=True)
    combinedchunks = find_compositeIntervals(df, which_status_to_check, 
                                            intervalLength=intervalLength,
                                            if_CombineAdjacentChunks=if_CombineAdjacentChunks,
                                            gapMax=gapMax)
    
    print(f"  {which_status_to_check}: found {len(combinedchunks)} intervals")
    
    df_Bonset_list = []
    df_onsetA_list = []
    
    if len(combinedchunks) > 0:
        # 打印更多信息
        if len(combinedchunks) > 0:
            print(f"  First interval: {combinedchunks[0]}, df length: {len(df)}")
        
        for interval in combinedchunks:
            if interval[0] >= secForward*fps:
                Bonset_start = interval[0] - secForward*fps
                Bonset_end = interval[0]
                onsetA_start = interval[0]
                if interval[0] + secBackward*fps < len(df):
                    onsetA_end = interval[0] + secBackward*fps
                    
                    # 检查是否真的能取到数据
                    if Bonset_start >= 0 and onsetA_end <= len(df):
                        Bonset_df = df_status.iloc[Bonset_start:Bonset_end]
                        Bonset_df = Bonset_df.drop('status', axis=1)
                        df_Bonset_list.append(Bonset_df)

                        onsetA_df = df_status.iloc[onsetA_start:onsetA_end]
                        onsetA_df = onsetA_df.drop('status', axis=1)
                        df_onsetA_list.append(onsetA_df)
                    else:
                        print(f"    Interval {interval} out of bounds")
                else:
                    print(f"    Interval {interval}: not enough frames after onset")
                    print(f"    Required: {interval[0] + secBackward*fps}, Available: {len(df)}")
        
    print(f"  Valid intervals found: {len(df_Bonset_list)} before, {len(df_onsetA_list)} after")
    
    # norm
    if ifNorm:
        df_Bonset_listN, df_onsetA_listN = normBA(df_status, df_Bonset_list, 
                                                df_onsetA_list, beforeB=beforeB, 
                                                baseSecond=baseSecond)
    else:
        df_Bonset_listN, df_onsetA_listN = df_Bonset_list, df_onsetA_list
        
    return (df_Bonset_listN, df_onsetA_listN)

# 修改版的average_nActEntries函数
def average_nActEntries_debug(df_list, neuron_list, fps, seconds):
    
    frames = seconds*fps
    avgNeuron_actEntries = pd.DataFrame()
    
    print(f"  average_nActEntries: {len(df_list)} dataframes, expecting {frames} frames")
    
    if len(df_list) > 0:
        for nCol in neuron_list:
            nAct_entries = []
            for i in range(len(df_list)):
                if df_list[i].shape[0] == frames:
                    nAct_entries.append(df_list[i][nCol].astype(float))
                else:
                    print(f"    DataFrame {i} has shape {df_list[i].shape}, expected {frames}")
            
            if len(nAct_entries) > 0:
                nAct_mean = np.mean(np.array(nAct_entries), axis=0)
                avgNeuron_actEntries[nCol] = nAct_mean
            else:
                print(f"    No valid entries for neuron {nCol}")
    
    print(f"  Result shape: {avgNeuron_actEntries.shape}")
    return avgNeuron_actEntries


# 设置主要参数

In [51]:

# 现在开始原代码的主要部分
print('Done')

# Path & Parameters
ROI = '260104'  
project = 'SocB'
session_ids = [4]   ###### session

root_path = f"F:/AAA-RXC"    # os.getcwd()   
data_path = os.path.join(root_path)  # , 'Data'
imgSave_path = os.path.join(root_path,'Results')

mice_list = ['PL1-4sessions', 'PL2-4sessions','PL4-4sessions','PL5-4sessions','PL6-4sessions','PL7-4sessions','PL8-4sessions']
mice_group = ['control']*len(mice_list)

# set parameters
# -------------------------------------------------------
# find_compositeIntervals()
gapMax = 10
intervalLength = 20
if_CombineAdjacentChunks = True

# normBA() 
ifNorm = False
baseSecond = 2
beforeB = False

# get_BonsetA()
secF = 3  # 定义所绘图中 event onset 之前的时间
secB = 8  # 定义所绘图中 event onset 之后的时间
ifFilterOutCenter = True
ifFilterOutCorner = False

fps = 15  # 数据记录时的frame rate 需注意！
amgColor = '#3333F5'
amgFillColor = '#aaaafa'

Done


In [52]:
# 分离不同的session
def separate_session(df, threshold=3):
    df = pd.read_csv(df, index_col=0)
    
    df.columns = [c.strip() for c in df.columns]  # 去除TRACE文件标题列的空格

    df['index_col'] = pd.to_numeric(df.index, errors='coerce')
    check_points = [0]  # 初始化第一个分割点为0（DataFrame的起始位置）
    frame_list = df['index_col'].values.tolist()

    # 判断两行之间的时间差，大于threshold则视为新的session
    for i in range(len(frame_list) - 1):
        if (frame_list[i+1] - frame_list[i]) > threshold:
            check_points.append(frame_list[i+1])
    
    check_points.append(frame_list[-1] + 1)  # 添加最后一个分割点

    session_dfs = []
    for i in range(len(check_points) - 1):
        session_df = df[(df['index_col'] >= check_points[i]) & (df['index_col'] < check_points[i+1])]
        session_df.drop(columns='index_col', inplace=True)
        session_dfs.append(session_df)

    return session_dfs

In [ ]:

# 导入其他需要的函数
from Functions_plot_All_20250807 import *

# 创建session数据
for i in range(len(mice_list)):
    
    mouseSummary = mice_list[i]
    mouseName = mouseSummary[-11:-7]   # mouse name selected from file name
    mouseType = mice_group[i]
    print("Processing ", mouseName+'-'+mouseType)
    
    trace = os.path.join(data_path, mouseSummary, "TRACE_Con.csv")
    df_traces = separate_session(trace)

    for session_id in session_ids:
        event = os.path.join(data_path, mouseSummary, f"event{session_id}.xlsx")
        
        session = df_traces[session_id-1]
        session.index.name = "Frame"
        session.index = pd.to_numeric(session.index)
        session_frame_label = add_event_label(session, event)

        session_frame_label.to_csv(os.path.join(data_path, mouseSummary, f"session_{session_id}_trace.csv"))

        print(project, f"session_{session_id}_trace has shape:", session_frame_label.shape)

    print('Done')
    session_frame_label.head(2)

# 主要处理逻辑
center_ceON_sessions = []
corner_ceON_sessions = []
sniffg_ceON_sessions = []
cs_ceON_sessions = []

for sessionID in session_ids:
    print('\nSession', sessionID)
    center_ceON = pd.DataFrame()
    corner_ceON = pd.DataFrame()
    sniffg_ceON = pd.DataFrame()
    cs_ceON = pd.DataFrame()
    
    for i in range(len(mice_list)):
        
        mouseSummary = mice_list[i]
        mouseName = mouseSummary[-13:-7]   # mouse name selected from file name
        mouseType = mice_group[i]

        event = os.path.join(data_path, mouseSummary, f"event{sessionID}.xlsx")
       
        df = pd.read_csv(os.path.join(data_path, mouseSummary, f"session_{sessionID}_trace.csv"))
        df.columns = [c.strip() for c in df.columns]  # 去除TRACE文件标题列的空格
        ONOFF = pd.read_excel(os.path.join(data_path, 'Results', mouseSummary, 
                                        f"session_{sessionID}_ONOFF.xlsx"))\
                                        .rename(columns={'Unnamed: 0': "Neurons"})\
                                        .set_index('Neurons')
        

        ON_list = ONOFF[ONOFF.cs_ON==1].index.to_list()  # the neuron type we interested ##########################################
        print(mouseSummary, 'session'+str(sessionID)+' with shape:', df.shape, 
              '| it has', len(ON_list), 'ON neurons') 

        # 步骤1: 从session中提取所有神经元列名 (排除Frame和Frame_Label)
        neuron_columns = df.columns[1:-1].tolist()  # 获取C001到C136等神经元列名

        # 步骤2: 筛选匹配的列
        # 找出在session中实际存在的ON神经元列
        valid_on_columns = [col for col in neuron_columns if col in ON_list]

        # 步骤3: 创建新的DataFrame (包含Frame, ON神经元和Frame_Label)
        new_df = df[["Frame"] + valid_on_columns + ["Frame_Label"]]

        # 验证结果
        print(f"原始ON_list神经元数量: {len(ON_list)}")
        print(f"session中匹配到的ON神经元数量: {len(valid_on_columns)}")
        print(f"New session shape: {new_df.shape}")

        if len(valid_on_columns) > 5:
    
            try:
                # 使用修改版的函数
                Bcenter_ceON_list, centerA_ceON_list = get_BonsetA_debug(
                    new_df, 'sniff', fps, secForward=secF, secBackward=secB,
                    beforeB=beforeB, baseSecond=baseSecond, intervalLength=intervalLength,
                    gapMax=gapMax, if_CombineAdjacentChunks=if_CombineAdjacentChunks,
                    ifNorm=ifNorm)
                
                Bcorner_ceON_list, cornerA_ceON_list = get_BonsetA_debug(
                    new_df, 'sniffed', fps, secF, secB, beforeB,
                    baseSecond, intervalLength, if_CombineAdjacentChunks, gapMax,
                    ifNorm=ifNorm)
                
                Bsniffg_ceON_list, sniffgA_ceON_list = get_BonsetA_debug(
                    new_df, 'freezing', fps, secF, secB, beforeB,
                    baseSecond, intervalLength, if_CombineAdjacentChunks, gapMax,
                    ifNorm=ifNorm)
                
                Bcs_ceON_list, csA_ceON_list = get_BonsetA_debug(
                    new_df, 'cs', fps, secF, secB, beforeB,
                    baseSecond, intervalLength, if_CombineAdjacentChunks, gapMax,
                    ifNorm=ifNorm)
                
                # 分别处理每个事件的数据
                # 1. sniff事件
                if len(Bcenter_ceON_list) > 0 and len(centerA_ceON_list) > 0:
                    Bcenter_ceON_Mean = average_nActEntries_debug(Bcenter_ceON_list, 
                                                                valid_on_columns, fps, secF)
                    centerA_ceON_Mean = average_nActEntries_debug(centerA_ceON_list, 
                                                                valid_on_columns, fps, secB)
                    BcenterA_ceON_Mean = pd.concat([Bcenter_ceON_Mean, centerA_ceON_Mean])
                    
                    if ifFilterOutCenter:
                        BcenterA_ceON_Mean = filterOut_nonceONce(BcenterA_ceON_Mean, fps, secF)
                    
                    if not BcenterA_ceON_Mean.empty:
                        center_ceON = pd.concat([center_ceON, BcenterA_ceON_Mean], axis=1)
                        print(f"  center shape: {center_ceON.shape}")
                
                # 2. sniffed事件
                if len(Bcorner_ceON_list) > 0 and len(cornerA_ceON_list) > 0:
                    Bcorner_ceON_Mean = average_nActEntries_debug(Bcorner_ceON_list, 
                                                                valid_on_columns, fps, secF)
                    cornerA_ceON_Mean = average_nActEntries_debug(cornerA_ceON_list, 
                                                                valid_on_columns, fps, secB)
                    BcornerA_ceON_Mean = pd.concat([Bcorner_ceON_Mean, cornerA_ceON_Mean])
                    
                    if ifFilterOutCorner:
                        BcornerA_ceON_Mean = filterOut_nonceONco(BcornerA_ceON_Mean, fps, secF)
                    
                    if not BcornerA_ceON_Mean.empty:
                        corner_ceON = pd.concat([corner_ceON, BcornerA_ceON_Mean], axis=1)
                        print(f"  corner shape: {corner_ceON.shape}")
                
                # 3. freezing事件
                if len(Bsniffg_ceON_list) > 0 and len(sniffgA_ceON_list) > 0:
                    Bsniffg_ceON_Mean = average_nActEntries_debug(Bsniffg_ceON_list, 
                                                                valid_on_columns, fps, secF)
                    sniffgA_ceON_Mean = average_nActEntries_debug(sniffgA_ceON_list, 
                                                                valid_on_columns, fps, secB)
                    BsniffgA_ceON_Mean = pd.concat([Bsniffg_ceON_Mean, sniffgA_ceON_Mean])
                    
                    if not BsniffgA_ceON_Mean.empty:
                        sniffg_ceON = pd.concat([sniffg_ceON, BsniffgA_ceON_Mean], axis=1)
                        print(f"  freezing shape: {sniffg_ceON.shape}")
                
                # 4. cs事件
                if len(Bcs_ceON_list) > 0 and len(csA_ceON_list) > 0:
                    Bcs_ceON_Mean = average_nActEntries_debug(Bcs_ceON_list, 
                                                            valid_on_columns, fps, secF)
                    csA_ceON_Mean = average_nActEntries_debug(csA_ceON_list, 
                                                            valid_on_columns, fps, secB)
                    BcsA_ceON_Mean = pd.concat([Bcs_ceON_Mean, csA_ceON_Mean])
                    
                    if not BcsA_ceON_Mean.empty:
                        cs_ceON = pd.concat([cs_ceON, BcsA_ceON_Mean], axis=1)
                        print(f"  cs shape: {cs_ceON.shape}")
                        
            except Exception as e:
                print(f"  处理过程中发生错误: {e}")
                import traceback
                traceback.print_exc()
                continue

    center_ceON = center_ceON.reset_index(drop=True)
    corner_ceON = corner_ceON.reset_index(drop=True)
    sniffg_ceON = sniffg_ceON.reset_index(drop=True)
    cs_ceON = cs_ceON.reset_index(drop=True)
    
    
    print(f'Session {sessionID} Done, with shapes:', 
          center_ceON.shape, corner_ceON.shape, 
          sniffg_ceON.shape, cs_ceON.shape)  
    print()

    # append to session list
    center_ceON_sessions.append(center_ceON)
    corner_ceON_sessions.append(corner_ceON)
    sniffg_ceON_sessions.append(sniffg_ceON)
    cs_ceON_sessions.append(cs_ceON)
    

print('Done')


Processing  1-4s-control
SocB session_4_trace has shape: (6302, 64)
Done
Processing  2-4s-control
SocB session_4_trace has shape: (6302, 68)
Done
Processing  4-4s-control
SocB session_4_trace has shape: (6303, 69)
Done
Processing  5-4s-control
SocB session_4_trace has shape: (6303, 52)
Done
Processing  6-4s-control
SocB session_4_trace has shape: (6302, 57)
Done
Processing  7-4s-control
SocB session_4_trace has shape: (6303, 103)
Done
Processing  8-4s-control
SocB session_4_trace has shape: (6302, 89)
Done

Session 4
PL1-4sessions session4 with shape: (6302, 65) | it has 25 ON neurons
原始ON_list神经元数量: 25
session中匹配到的ON神经元数量: 24
New session shape: (6302, 26)
[[1171, 1320], [5554, 5688]]
  sniff: found 2 intervals
  First interval: [1171, 1320], df length: 6302
  Valid intervals found: 2 before, 2 after
   list index out of range
no results
  sniffed: found 0 intervals
  Valid intervals found: 0 before, 0 after
[[151, 180], [571, 600], [856, 915], [961, 990], [1426, 1455], [1802, 1831], [

In [54]:
# Plot Heatmap & Average Trace
cmap = 'coolwarm'  # 'coolwarm' 'seismic'
ifCenter = False

for i in range(len(session_ids)):

    sessionID = session_ids[i]
    print(f"\n处理Session {sessionID}")
    
    center_ceON = center_ceON_sessions[i]
    corner_ceON = corner_ceON_sessions[i]
    sniffg_ceON = sniffg_ceON_sessions[i]
    cs_ceON = cs_ceON_sessions[i]
    
    # 检查数据
    print("检查数据：")
    print(f"center_ceON 类型: {type(center_ceON)}, 形状: {center_ceON.shape}")
    print(f"corner_ceON 类型: {type(corner_ceON)}, 形状: {corner_ceON.shape}")
    print(f"sniffg_ceON 类型: {type(sniffg_ceON)}, 形状: {sniffg_ceON.shape}")
    print(f"cs_ceON 类型: {type(cs_ceON)}, 形状: {cs_ceON.shape}")
    
    # 处理每个数据，添加错误处理
    normed_center_ceON = pd.DataFrame()
    normed_corner_ceON = pd.DataFrame()
    normed_sniffg_ceON = pd.DataFrame()
    normed_cs_ceON = pd.DataFrame()
    
    try:
        print("处理center_ceON...")
        if not center_ceON.empty:
            normed_center_ceON = mixMaxScale_dff(center_ceON, (-1, 4))
            print("数据处理成功")
        else:
            print("center_ceON 数据为空")
    except Exception as e:
        print(f"处理center_ceON失败: {e}")
    
    try:
        print("处理corner_ceON...")
        if not corner_ceON.empty:
            normed_corner_ceON = mixMaxScale_dff(corner_ceON, (-1, 4))
            print("数据处理成功")
        else:
            print("corner_ceON 数据为空")
    except Exception as e:
        print(f"处理corner_ceON失败: {e}")
    
    try:
        print("处理sniffg_ceON...")
        if not sniffg_ceON.empty:
            normed_sniffg_ceON = mixMaxScale_dff(sniffg_ceON, (-1, 4))
            print("数据处理成功")
        else:
            print("sniffg_ceON 数据为空")
    except Exception as e:
        print(f"处理sniffg_ceON失败: {e}")
    
    try:
        print("处理cs_ceON...")
        if not cs_ceON.empty:
            normed_cs_ceON = mixMaxScale_dff(cs_ceON, (-1, 4))
            print("数据处理成功")
        else:
            print("cs_ceON 数据为空")
    except Exception as e:
        print(f"处理cs_ceON失败: {e}")
    
    # 改进：对神经元进行排序，按照它们的峰值时间
    def sort_neurons_by_peak_time(df):
        """
        按照神经元的峰值时间对列进行排序
        使得峰值时间早的神经元在前，峰值时间晚的神经元在后
        """
        if df.empty:
            return df
        
        # 找到每个神经元的峰值时间（最大值的索引）
        peak_times = {}
        for col in df.columns:
            # 找到最大值的索引（如果有多个最大值，取第一个）
            max_val = df[col].max()
            # 找到所有最大值的位置
            peak_indices = df.index[df[col] == max_val].tolist()
            if peak_indices:
                # 如果有多个峰值，取第一个
                peak_times[col] = peak_indices[0]
            else:
                # 如果没有找到峰值，使用中间位置
                peak_times[col] = len(df) // 2
        
        # 按照峰值时间排序
        sorted_columns = sorted(peak_times.items(), key=lambda x: x[1])
        sorted_columns = [col for col, _ in sorted_columns]
        
        # 重新排列DataFrame的列
        return df[sorted_columns]
    
    def sort_neurons_by_weighted_peak_time(df, window_size=3):
        """
        使用加权方法计算神经元的峰值时间，考虑峰值周围的区域
        使得排序更平滑
        """
        if df.empty:
            return df
        
        peak_scores = {}
        
        for col in df.columns:
            # 获取神经元的活性数据
            activity = df[col].values
            
            # 计算加权峰值分数
            # 权重：当前位置和附近位置的加权和
            weighted_scores = []
            for j in range(len(activity)):
                # 计算以j为中心的窗口内的加权和
                start = max(0, j - window_size)
                end = min(len(activity), j + window_size + 1)
                window_sum = np.sum(activity[start:end])
                weighted_scores.append(window_sum)
            
            # 找到加权分数最高的位置
            peak_idx = np.argmax(weighted_scores)
            peak_scores[col] = peak_idx
        
        # 按照峰值位置排序
        sorted_columns = sorted(peak_scores.items(), key=lambda x: x[1])
        sorted_columns = [col for col, _ in sorted_columns]
        
        return df[sorted_columns]
    
    # 应用排序
    if not normed_center_ceON.empty:
        print("对center_ceON神经元进行排序...")
        normed_center_ceON = sort_neurons_by_weighted_peak_time(normed_center_ceON, window_size=5)
    
    if not normed_corner_ceON.empty:
        print("对corner_ceON神经元进行排序...")
        normed_corner_ceON = sort_neurons_by_weighted_peak_time(normed_corner_ceON, window_size=5)
    
    if not normed_sniffg_ceON.empty:
        print("对sniffg_ceON神经元进行排序...")
        normed_sniffg_ceON = sort_neurons_by_weighted_peak_time(normed_sniffg_ceON, window_size=5)
    
    if not normed_cs_ceON.empty:
        print("对cs_ceON神经元进行排序...")
        normed_cs_ceON = sort_neurons_by_weighted_peak_time(normed_cs_ceON, window_size=5)

    # 检查数据是否为空
    if normed_center_ceON.empty:
        print("警告: center_ceON 数据为空，跳过heatmap绘制")
    else:
        # heatmap ------------------------------------------------
        # 添加额外的信息：显示排序后的神经元顺序
        print(f"排序后的神经元顺序 (center): {list(normed_center_ceON.columns)}")
        
        # 绘制heatmap
        plot_colomap(normed_center_ceON, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path, 
                     filename='heatmap_session'+str(sessionID)+'_ON_sniff_heatmap_sorted')
        
        # 同时保存未排序的版本用于对比
        try:
            if not center_ceON.empty:
                normed_center_ceON_unsorted = mixMaxScale_dff(center_ceON, (-1, 4))
                plot_colomap(normed_center_ceON_unsorted, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                             ifSave=True, savePath=imgSave_path, 
                             filename='heatmap_session'+str(sessionID)+'_ON_sniff_heatmap_unsorted')
        except:
            pass
    
    if normed_corner_ceON.empty:
        print("警告: corner_ceON 数据为空，跳过heatmap绘制")
    else:
        # 添加额外的信息：显示排序后的神经元顺序
        print(f"排序后的神经元顺序 (corner): {list(normed_corner_ceON.columns)}")
        
        plot_colomap(normed_corner_ceON, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path, 
                     filename='heatmap_session'+str(sessionID)+'_ON_sniffed_heatmap_sorted')
        
        # 同时保存未排序的版本用于对比
        try:
            if not corner_ceON.empty:
                normed_corner_ceON_unsorted = mixMaxScale_dff(corner_ceON, (-1, 4))
                plot_colomap(normed_corner_ceON_unsorted, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                             ifSave=True, savePath=imgSave_path, 
                             filename='heatmap_session'+str(sessionID)+'_ON_sniffed_heatmap_unsorted')
        except:
            pass
    
    if normed_sniffg_ceON.empty:
        print("警告: sniffg_ceON 数据为空，跳过heatmap绘制")
    else:
        # 添加额外的信息：显示排序后的神经元顺序
        print(f"排序后的神经元顺序 (freezing): {list(normed_sniffg_ceON.columns)}")
        
        plot_colomap(normed_sniffg_ceON, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path, 
                     filename='heatmap_session'+str(sessionID)+'_ON_freezing_heatmap_sorted')
        
        # 同时保存未排序的版本用于对比
        try:
            if not sniffg_ceON.empty:
                normed_sniffg_ceON_unsorted = mixMaxScale_dff(sniffg_ceON, (-1, 4))
                plot_colomap(normed_sniffg_ceON_unsorted, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                             ifSave=True, savePath=imgSave_path, 
                             filename='heatmap_session'+str(sessionID)+'_ON_freezing_heatmap_unsorted')
        except:
            pass
    
    if normed_cs_ceON.empty:
        print("警告: cs_ceON 数据为空，跳过heatmap绘制")
    else:
        # 添加额外的信息：显示排序后的神经元顺序
        print(f"排序后的神经元顺序 (cs): {list(normed_cs_ceON.columns)}")
        
        plot_colomap(normed_cs_ceON, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path, 
                     filename='heatmap_session'+str(sessionID)+'_ON_cs_heatmap_sorted')
        
        # 同时保存未排序的版本用于对比
        try:
            if not cs_ceON.empty:
                normed_cs_ceON_unsorted = mixMaxScale_dff(cs_ceON, (-1, 4))
                plot_colomap(normed_cs_ceON_unsorted, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                             ifSave=True, savePath=imgSave_path, 
                             filename='heatmap_session'+str(sessionID)+'_ON_cs_heatmap_unsorted')
        except:
            pass

    # average trace ------------------------------------------------
    if not normed_center_ceON.empty:
        normed_center_ceON_summary = getSummary(normed_center_ceON)
        x, WT, WT_SEM, WTSEM, vmin, vmax = get_minMaxForPlot_One(normed_center_ceON_summary)
        plotAgeTrace_One(x, WT, WT_SEM, WTSEM, vmin, vmax, xVline=secF*fps, 
                         cWT=amgColor, cFWT=amgFillColor, 
                         ifSave=True, savePath=imgSave_path, 
                         filename='avgTrace_session'+str(sessionID)+'ON_sniff')
    
    if not normed_corner_ceON.empty:
        normed_corner_ceON_summary = getSummary(normed_corner_ceON)
        x, WT, WT_SEM, WTSEM, vmin, vmax = get_minMaxForPlot_One(normed_corner_ceON_summary)
        plotAgeTrace_One(x, WT, WT_SEM, WTSEM, vmin-0.3, vmax, xVline=secF*fps, 
                         cWT=amgColor, cFWT=amgFillColor, 
                         ifSave=True, savePath=imgSave_path, 
                         filename='avgTrace_session'+str(sessionID)+'ON_sniffed')
    
    if not normed_sniffg_ceON.empty:
        normed_sniffg_ceON_summary = getSummary(normed_sniffg_ceON)
        x, WT, WT_SEM, WTSEM, vmin, vmax = get_minMaxForPlot_One(normed_sniffg_ceON_summary)
        plotAgeTrace_One(x, WT, WT_SEM, WTSEM, vmin, vmax, xVline=secF*fps, 
                         cWT=amgColor, cFWT=amgFillColor, 
                         ifSave=True, savePath=imgSave_path, 
                         filename='avgTrace_session'+str(sessionID)+'ON_freezing')
    
    if not normed_cs_ceON.empty:
        normed_cs_ceON_summary = getSummary(normed_cs_ceON)
        x, WT, WT_SEM, WTSEM, vmin, vmax = get_minMaxForPlot_One(normed_cs_ceON_summary)
        plotAgeTrace_One(x, WT, WT_SEM, WTSEM, vmin, vmax, xVline=secF*fps, 
                         cWT=amgColor, cFWT=amgFillColor, 
                         ifSave=True, savePath=imgSave_path, 
                         filename='avgTrace_session'+str(sessionID)+'ON_cs')
    
    print(f"图像保存路径: {imgSave_path}")


处理Session 4
检查数据：
center_ceON 类型: <class 'pandas.core.frame.DataFrame'>, 形状: (165, 53)
corner_ceON 类型: <class 'pandas.core.frame.DataFrame'>, 形状: (0, 0)
sniffg_ceON 类型: <class 'pandas.core.frame.DataFrame'>, 形状: (165, 114)
cs_ceON 类型: <class 'pandas.core.frame.DataFrame'>, 形状: (165, 114)
处理center_ceON...
数据处理成功
处理corner_ceON...
corner_ceON 数据为空
处理sniffg_ceON...
数据处理成功
处理cs_ceON...
数据处理成功
对center_ceON神经元进行排序...
对sniffg_ceON神经元进行排序...
对cs_ceON神经元进行排序...
排序后的神经元顺序 (center): ['C062', 'C030', 'C036', 'C072', 'C143', 'C053', 'C165', 'C017', 'C174', 'C103', 'C103', 'C028', 'C022', 'C116', 'C163', 'C063', 'C004', 'C056', 'C001', 'C060', 'C066', 'C043', 'C043', 'C006', 'C133', 'C040', 'C157', 'C156', 'C055', 'C128', 'C026', 'C068', 'C046', 'C131', 'C122', 'C130', 'C140', 'C123', 'C084', 'C070', 'C058', 'C061', 'C033', 'C027', 'C027', 'C027', 'C016', 'C045', 'C037', 'C083', 'C149', 'C057', 'C059']
警告: corner_ceON 数据为空，跳过heatmap绘制
排序后的神经元顺序 (freezing): ['C057', 'C073', 'C002', 'C059', 'C059', 'C